In [1]:
# Import necessary libraries
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from warnings import filterwarnings
filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
# Load the dataset
raw_data = pd.read_csv('Data/postings.csv')
df = raw_data.copy()
df.head()

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,17.0,Full-time,2.0,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/921716/?trk...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,Requirements: \n\nWe are seeking a College or ...,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,30.0,Full-time,NaN,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/1829192/?tr...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,NaN,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,45000.0,Full-time,NaN,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/10998357/?t...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,We are currently accepting resumes for FOH - A...,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,140000.0,Full-time,NaN,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/23221523/?t...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,This position requires a baseline understandin...,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,60000.0,Full-time,NaN,1.710000e+12,NaN,https://www.linkedin.com/jobs/view/35982263/?t...,NaN,ComplexOnsiteApply,1.720000e+12,NaN,NaN,NaN,1.710000e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


In [3]:
# Explore the dataset
print(f"Rows: {df.shape[0]}")
print("-"*20)
print(f"\nColumns: {df.shape[1]}")
print("-"*20)
print(f"\nColumns: {df.columns.tolist()}")

Rows: 123849
--------------------

Columns: 31
--------------------

Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  str    
 2   title                       123849 non-null  str    
 3   description                 123842 non-null  str    
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   str    
 6   location                    123849 non-null  str    
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  str    
 12  applies                     23320 non-null   float64
 13  original_listed_time     

In [5]:
# Fix data types
count_cols = ['company_id', 'views', 'applies', 'remote_allowed']
for col in count_cols:
    df[col] = df[col].astype('Int64')

time_cols = ['original_listed_time', 'expiry', 'closed_time', 'listed_time']
for col in time_cols:
    df[col] = pd.to_datetime(df[col], unit='ms', errors='coerce')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   job_id                      123849 non-null  int64         
 1   company_name                122130 non-null  str           
 2   title                       123849 non-null  str           
 3   description                 123842 non-null  str           
 4   max_salary                  29793 non-null   float64       
 5   pay_period                  36073 non-null   str           
 6   location                    123849 non-null  str           
 7   company_id                  122132 non-null  Int64         
 8   views                       122160 non-null  Int64         
 9   med_salary                  6280 non-null    float64       
 10  min_salary                  29793 non-null   float64       
 11  formatted_work_type         123849 non-null  str  

In [6]:
df.sample(10)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
46090,3900963120,Aditi Consulting,Snowflake/AWS Integration Architect,Job Title: IT Software Engineer 5 - Snowflake/...,100.0,HOURLY,"Illinois, United States",2985733,18,NaN,85.0,Contract,2,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3900963120/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,NaN,0,CONTRACT,USD,BASE_SALARY,192400.0,NaN,NaN
115559,3905852294,Compunnel Inc.,Procurement Specialist,"Contract PositionHybrid rolePhoenix, AZ / Temp...",NaN,NaN,"Phoenix, AZ",16690,1,NaN,NaN,Contract,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3905852294/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,NaN,0,CONTRACT,NaN,NaN,NaN,85003.0,4013.0
10467,3886895657,Aquent Talent,Front End Engineer (Javascript),"Our client, a leading consulting firm, is look...",NaN,NaN,"Irving, TX",2217719,172,NaN,NaN,Contract,90,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3886895657/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,NaN,0,CONTRACT,NaN,NaN,NaN,75038.0,48113.0
41158,3899519969,Pantheon,Senior Legal Associate,Senior AssociatePrivate Credit Investment Stru...,NaN,NaN,"New York, United States",23934,47,NaN,NaN,Full-time,9,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3899519969/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,NaN,0,FULL_TIME,NaN,NaN,NaN,NaN,NaN
63005,3902342846,MAU Workforce Solutions,Manufacturing Engineer,MAU is hiring a Manufacturing Engineer at our ...,NaN,NaN,"Greenville, SC",35074,2,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3902342846/...,NaN,ComplexOnsiteApply,2024-07-03 09:46:40,NaT,Associate,NaN,2024-03-09 16:00:00,NaN,0,FULL_TIME,NaN,NaN,NaN,29601.0,45045.0
74801,3903435346,"North Shore Healthcare, LLC",Registered Nurse (RN) - Full-Time All Shifts,Position Details\n\nJob Title Shift: All Shift...,NaN,NaN,"Deerfield, WI",16182319,4,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3903435346/...,NaN,SimpleOnsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,nshorehc.hcshiring.com,0,FULL_TIME,NaN,NaN,NaN,53531.0,55025.0
53734,3901666729,Milwaukee Tool,Repair Technician / Service Coordinator,Job Description\n\nJob Description\n\nUnder th...,NaN,NaN,"Dallas, TX",164989,10,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3901666729/...,https://tti.wd1.myworkdayjobs.com/Milwaukee/jo...,OffsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,tti.wd1.myworkdayjobs.com,0,FULL_TIME,NaN,NaN,NaN,75201.0,48113.0
96660,3904950327,Dice,Senior Structural Engineer Department Head,Dice is the leading career destination for tec...,NaN,NaN,"Melbourne, FL",6849,3,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904950327/...,https://click.appcast.io/track/j7vd24f?cs=ivj&...,OffsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,click.appcast.io,0,FULL_TIME,NaN,NaN,NaN,32901.0,12009.0
117452,3905886931,"Quantix, Inc.",Enterprise Resources Planning Implementation C...,"Title: ERP Implementation Resource, JuniorLoca...",38.0,HOURLY,"Dallas, GA",912694,4,NaN,35.0,Contract,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3905886931/...,NaN,ComplexOnsiteApply,2024-10-27 03:33:20,NaT,Associate,NaN,2024-03-09 16:00:00,NaN,0,CONTRACT,USD,BASE_SALARY,75920.0,30132.0,13223.0
54228,3901674522,Guthrie,Housekeeper-Acute Care PA

In [7]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.sample(10)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
82846,3904040356,Energy Jobline,"CDL A Truck Driver - Home Weekly - Earn $1,690...",Job Description\n\nCDL A Truck Driver – Home W...,NaN,NaN,"Buffalo, NY",2379156,2,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904040356/...,https://www.energyjobline.com/job/cdl-truck-dr...,OffsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,www.energyjobline.com,0,FULL_TIME,NaN,NaN,NaN,14201.0,36029.0
90133,3904420117,Safeway,Starbucks Barista,"A Day in the Life:\n\nAs a Barista, you will b...",NaN,NaN,"Alexandria, VA",3498,6,NaN,NaN,Part-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3904420117/...,https://eofd.fa.us6.oraclecloud.com/hcmUI/Cand...,OffsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,eofd.fa.us6.oraclecloud.com,0,PART_TIME,NaN,NaN,NaN,22301.0,51510.0
51123,3901379130,PwC,External Audit Senior Associate - Technology a...,Specialty/Competency: Assurance\n\nIndustry/Se...,NaN,NaN,"Dallas, TX",1044,3,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,1,https://www.linkedin.com/jobs/view/3901379130/...,https://ad.doubleclick.net/ddm/clk/438405794;2...,OffsiteApply,2024-07-03 09:46:40,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,jobs.us.pwc.com,0,FULL_TIME,NaN,NaN,NaN,75201.0,48113.0
2696,3884810139,Pacific Guardian Life,Customer Service Representative,As a Customer Service Representative with Paci...,25.0,HOURLY,"Honolulu, HI",1910013,2,NaN,20.0,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3884810139/...,NaN,ComplexOnsiteApply,2024-03-09 16:00:00,NaT,NaN,NaN,2024-03-09 16:00:00,NaN,0,FULL_TIME,USD,BASE_SALARY,46800.0,96813.0,15003.0
112737,3905660517,Johnson Controls,HVAC Customer Support Coordinator,Unleash your potential with the Johnson Contro...,NaN,NaN,"Metairie, LA",2247,4,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3905660517/...,https://click.appcast.io/track/j8697by-org?cs=...,OffsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,click.appcast.io,0,FULL_TIME,NaN,NaN,NaN,70001.0,22051.0
12952,3887710940,"Brandon J. Broderick, Personal Injury Attorney...",Communications/PR Consultant,"About Brandon J. Broderick, Attorney At Law:Br...",100.0,HOURLY,"River Edge, NJ",60813588,81,NaN,80.0,Full-time,11,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3887710940/...,NaN,ComplexOnsiteApply,2024-10-27 03:33:20,NaT,NaN,NaN,2024-03-09 16:00:00,NaN,0,FULL_TIME,USD,BASE_SALARY,187200.0,7661.0,34003.0
81748,3903874905,TEKsystems,Onsite Support Technician,Working 100% onsite in Kansas City MO \n\n ﻿De...,NaN,NaN,"Kansas City, MO",2152,4,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3903874905/...,https://jsv3.recruitics.com/redirect?rx_cid=34...,OffsiteApply,2024-07-03 09:46:40,NaT,Entry level,NaN,2024-03-09 16:00:00,jsv3.recruitics.com,0,FULL_TIME,NaN,NaN,NaN,64101.0,29095.0
11242,3887484020,Dollar Tree Stores,STORE MANAGER,Store Dollar Tree\n\nWork where you love to sh...,NaN,NaN,"Chelsea, MA",18613,4,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3887484020/...,https://careers.dollartree.com/us/en/job/DTYDT...,OffsiteApply,2024-03-09 16:00:00,NaT,Mid-Senior level,NaN,2024-03-09 16:00:00,careers.dollartree.com,0,FULL_TIME,NaN,NaN,NaN,2150.0,25025.0
21596,3889444654,Goodwin Recruiting,Medical Device RN (ICU),To Apply for this Job Click Here\n\nThe Goodwi...,NaN,NaN,"Chicago, IL",74985,4,NaN,NaN,Full-time,<NA>,2024-03-09 16:00:00,<NA>,https://www.linkedin.com/jobs/view/3889444654/...,https://www.g

In [8]:
# check for duplicates
df.duplicated().sum()

np.int64(0)

In [9]:
df[df.duplicated(subset=['job_id'], keep=False)]

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips


In [10]:
# Check for missing values
df.isnull().sum()

job_id                             0
company_name                    1719
title                              0
description                        7
max_salary                     94056
pay_period                     87776
location                           0
company_id                      1717
views                           1689
med_salary                    117569
min_salary                     94056
formatted_work_type                0
applies                       100529
original_listed_time               0
remote_allowed                108603
job_posting_url                    0
application_url                36665
application_type                   0
expiry                             0
closed_time                   122776
formatted_experience_level     29409
skills_desc                   121410
listed_time                        0
posting_domain                 39968
sponsored                          0
work_type                          0
currency                       87776
c

## Handle missing values

In [13]:
df['company_name'] = df['company_name'].fillna('Unknown')

df['company_id'] = df['company_id'].astype('object').fillna('Unknown').astype(str)

# Drop null values from description
df.dropna(subset=['description'], inplace=True)

df['remote_allowed'] = df['remote_allowed'].fillna(0)
df['applies'] = df['applies'].fillna(0)
df['views'] = df['views'].fillna(0)

df['formatted_experience_level'] = df['formatted_experience_level'].fillna('Not Specified')
df['posting_domain'] = df['posting_domain'].fillna('Unknown')

df['has_salary_listed'] = df['normalized_salary'].notna().astype(int)

In [14]:
columns_to_drop = [
        'skills_desc', 
        'closed_time', 
        'fips', 
        'zip_code',
        'max_salary', 
        'min_salary', 
        'med_salary', 
        'pay_period', 
        'currency', 
        'compensation_type',
        'job_posting_url',
        'application_url'
    ]
    
df.drop(columns=columns_to_drop, errors='ignore', inplace=True)

In [15]:
# Date features
df['listing_year'] = df['listed_time'].dt.year
df['listing_month'] = df['listed_time'].dt.month
df['listing_day'] = df['listed_time'].dt.day

In [16]:
# cleaning description column

from typing import Any


import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set[Any](stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def nlp_cleaner(text):
    if not isinstance(text, str):
        return ""

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

     # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)  
    
    # Remove newlines and carriage returns
    text = re.sub(r'[\n\r]', ' ', text)

    # Remove special characters (keep only alphanumeric)
    text = re.sub(r'[^a-z0-9 ]', ' ', text) 



    # text.split() handles tokenization AND naturally drops multiple spaces (\s+)
    cleaned_words = [
        lemmatizer.lemmatize(word) for word in text.split() if word not in stop_words
    ]
    
    # Rejoin the tokens into a single clean string
    return " ".join(cleaned_words)

# Apply it to the dataframe
df['clean_description'] = df['description'].apply(nlp_cleaner)

In [17]:
# Exteact skills
skills = [
    'python',
    'sql',
    'power bi',
    'tableau',
    'excel',
    'aws',
    'azure',
    'gcp',
    'machine learning',
    'deep learning',
    'pandas',
    'numpy',
    'spark',
    'hadoop'
]

def extract_skills(text):

    found = []

    text = str(text).lower()

    for skill in skills:
        if skill in text:
            found.append(skill)

    return found

df['skills'] = (
    df['clean_description']
    .apply(extract_skills)
)


df['skill_count'] = (
    df['skills']
    .apply(len)
)

In [18]:
df.sample(50)

,job_id,company_name,title,description,location,company_id,views,formatted_work_type,applies,original_listed_time,remote_allowed,application_type,expiry,formatted_experience_level,listed_time,posting_domain,sponsored,work_type,normalized_salary,has_salary_listed,listing_year,listing_month,listing_day,clean_description,skills,skill_count
21449,3889436502,"Lowe's Companies, Inc.",Full Time - Cashier - Opening,Essential Functions:\n\nNOTE: Minors in this r...,"Dallas, TX",4128,5,Full-time,0,2024-03-09 16:00:00,0,OffsiteApply,2024-07-03 09:46:40,Entry level,2024-03-09 16:00:00,talent.lowes.com,0,FULL_TIME,NaN,0,2024,3,9,ssential unction inors role may responsible ac...,[],0
52896,3901630026,American Family Insurance,"Insurance Agency Owner - Mount Pleasant, IA",Business owner. Community leader. Protector of...,"Mount Pleasant, IA",4647,6,Full-time,0,2024-03-09 16:00:00,0,OffsiteApply,2024-07-03 09:46:40,Executive,2024-03-09 16:00:00,amfam.wd1.myworkdayjobs.com,0,FULL_TIME,NaN,0,2024,3,9,usiness owner ommunity leader rotector dream h...,[],0
43729,3900081436,Tata Consultancy Services,Functional Analyst,Execution of multiple user stories to validate...,"Pleasanton, CA",1353,29,Full-time,4,2024-03-09 16:00:00,0,OffsiteApply,2024-07-03 09:46:40,Mid-Senior level,2024-03-09 16:00:00,ibegin.tcs.com,0,FULL_TIME,NaN,0,2024,3,9,xecution multiple user story validate differen...,[],0
30088,3894547434,Northside Hospital,Front Office Assistant PRN - Midtown Medical A...,2024-61352\n\nNorthside Hospital is award-winn...,"Atlanta, GA",16182,4,Full-time,0,2024-03-09 16:00:00,0,OffsiteApply,2024-07-03 09:46:40,Not Specified,2024-03-09 16:00:00,NorthsideHospital.contacthr.com,0,FULL_TIME,NaN,0,2024,3,9,2024 61352 orthside ospital award winning stat...,[],0
116805,3905874734,Hunter International Recruiting,Digital Marketing Specialist,"Toledo, OH – Hybrid \n\n Pay Rate: $33/HR \n\n...","Toledo, OH",6631661,3,Full-time,0,2024-03-09 16:00:00,0,OffsiteApply,2024-07-03 09:46:40,Entry level,2024-03-09 16:00:00,www.hirecruiting.com,0,FULL_TIME,NaN,0,2024,3,9,oledo ybrid ay ate 33 igital arketing pecialis...,[],0
123837,3906265414,"TalentBurst, an Inc 5000 company",Contract Administrator,"Position: Clinical Contracts Analyst, Req#: 63...","Irvine, CA",122451,1,Contract,0,2024-03-09 16:00:00,0,ComplexOnsiteApply,2024-07-03 09:46:40,Mid-Senior level,2024-03-09 16:00:00,Unknown,0,CONTRACT,83200.0,1,2024,3,9,osition linical ontracts nalyst eq 6351 1 ocat...,[],0
118984,3906221640,Easterseals-Goodwill Northern Rocky Mountain,Donation Door Attendant,TEXT ‘GoodwillJobs’ to 314-665-1767 to apply\n...,"Murray, UT",822191,3,Part-time,0,2024-03-09 16:00:00,0,OffsiteApply,2024-07-03 09:46:40,Entry level,2024-03-09 16:00:00,recruiting2.ultipro.com,0,PART_TIME,NaN,0,2024,3,9,oodwill ob 314 665 1767 apply pply onation oor...,[],0
54164,3901673725,Pride Health,Phlebotomist,Pride Health is currently looking candidates f...,"Leesburg, FL",2213829,20,Contract,0,2024-03-09 16:00:00,0,ComplexOnsiteApply,2024-07-03 09:46:40,Associate,2024-03-09 16:00:00,Unknown,0,CONTRACT,35360.0,1,2024,3,9,ride ealth currently looking candidate hleboto...,[],0
69116,3902791043,"Community Options, Inc.",Employee Relations Manager,"Community Options, Inc. is a national non-prof...","Princeton, NJ",1096036,8,Full-time,0,2024-03-09 16:00:00,0,OffsiteApply,2024-07-03 09:46:40,Mid-Senior level,2024-03-09 16:00:00,recruiting.ultipro.com,0,FULL_TIME,NaN,0,2024,3,9,ommunity ptions nc national non profit agency ...,[],0
119857,3906226724,Warner Bros. Discovery,Producer,"Every great story has a new beginning, and you...","Burbank, CA",73234923,3,Full-time,0,2024-03-09 16:00:00,0,OffsiteApply,2024-07-03 09:46:40,Mid-Senior level,2024-03-09 16:00:00,careers.wbd.com,0,FULL_TIME,89700.0,1,2024,3,9,great story new beginning start elcome arner r...,[],0


In [ ]:
import pandas as pd

job_skills = pd.read_csv('Data/jobs/job_skills.csv')
skills = pd.read_csv('Data/mappings/skills.csv')
job_industries = pd.read_csv('Data/jobs/job_industries.csv')
industries = pd.read_csv('Data/mappings/industries.csv')
companies = pd.read_csv('Data/companies/companies.csv')


display(job_skills)
print("-"*20)
display(skills)
print("-"*20)
display(job_industries)
print("-"*20)
display(industries)
print("-"*20)
display(companies)

,job_id,skill_abr
0,3884428798,MRKT
1,3884428798,PR
2,3884428798,WRT
3,3887473071,SALE
4,3887465684,FIN
...,...,...
213763,3902876855,HR
213764,3902878689,MGMT
213765,3902878689,MNFC
213766,3902883233,SALE


--------------------


,skill_abr,skill_name
0,ART,Art/Creative
1,DSGN,Design
2,ADVR,Advertising
3,PRDM,Product Management
4,DIST,Distribution
5,EDU,Education
6,TRNG,Training
7,PRJM,Project Management
8,CNSL,Consulting
9,PRCH,Purchasing


--------------------


,job_id,industry_id
0,3884428798,82
1,3887473071,48
2,3887465684,41
3,3887467939,82
4,3887467939,80
...,...,...
164803,3902882321,104
164804,3902879720,27
164805,3902876855,80
164806,3902878689,116


--------------------


,industry_id,industry_name
0,1,Defense and Space Manufacturing
1,3,Computer Hardware Manufacturing
2,4,Software Development
3,5,Computer Networking Products
4,6,"Technology, Information and Internet"
...,...,...
417,3249,Surveying and Mapping Services
418,3250,Retail Pharmacies
419,3251,Climate Technology Product Manufacturing
420,3252,Climate Data and Analytics


--------------------


,company_id,name,description,company_size,state,country,city,zip_code,address,url
0,1009,IBM,"At IBM, we do more than work. We create. We cr...",7.0,NY,US,"Armonk, New York",10504,International Business Machines Corp.,https://www.linkedin.com/company/ibm
1,1016,GE HealthCare,Every day millions of people feel the impact o...,7.0,0,US,Chicago,0,-,https://www.linkedin.com/company/gehealthcare
2,1025,Hewlett Packard Enterprise,Official LinkedIn of Hewlett Packard Enterpris...,7.0,Texas,US,Houston,77389,1701 E Mossy Oaks Rd Spring,https://www.linkedin.com/company/hewlett-packa...
3,1028,Oracle,We’re a cloud technology company that provides...,7.0,Texas,US,Austin,78741,2300 Oracle Way,https://www.linkedin.com/company/oracle
4,1033,Accenture,Accenture is a leading global professional ser...,7.0,0,IE,Dublin 2,0,Grand Canal Harbour,https://www.linkedin.com/company/accenture
...,...,...,...,...,...,...,...,...,...,...
24468,103463217,JRC Services,NaN,2.0,0,0,0,0,0,https://www.linkedin.com/company/jrcservices
24469,103466352,Centent Consulting LLC,Centent Consulting LLC is a reputable human re...,NaN,0,0,0,0,0,https://www.linkedin.com/company/centent-consu...
24470,103467540,"Kings and Queens Productions, LLC",We are a small but mighty collection of thinke...,NaN,0,0,0,0,0,https://www.linkedin.com/company/kings-and-que...
24471,103468936,WebUnite,Our mission at WebUnite is to offer experience...,NaN,Pennsylvania,US,Southampton,18966,720 2nd Street Pike,https://www.linkedin.com/company/webunite


In [22]:
datasets = {
    "Job Skills": job_skills,
    "Skills Mapping": skills,
    "Job Industries": job_industries,
    "Industries Mapping": industries,
    "Companies": companies
}

def generate_quality_report(data_dict):
    report = []
    
    for name, df in data_dict.items():
        total_rows = len(df)
        total_cols = len(df.columns)
        duplicates = df.duplicated().sum()
        
        # missing values 
        null_counts = df.isnull().sum()
        missing_cols = [f"{col} ({count})" for col, count in null_counts.items() if count > 0]
        missing_str = ", ".join(missing_cols) if missing_cols else "No Missing Data"
        
        # Append to report
        report.append({
            "Dataset Name": name,
            "Rows": f"{total_rows:,}",
            "Columns": total_cols,
            "Duplicate Rows": f"{duplicates:,}",
            "Columns w/ Nulls (Count)": missing_str
        })
        
    return pd.DataFrame(report)

# 3. Generate and display the clean summary
print("--- DATA QUALITY REPORT ---")
dq_report = generate_quality_report(datasets)
display(dq_report)

--- DATA QUALITY REPORT ---


,Dataset Name,Rows,Columns,Duplicate Rows,Columns w/ Nulls (Count)
0,Job Skills,"213,768",2,0,No Missing Data
1,Skills Mapping,35,2,0,No Missing Data
2,Job Industries,"164,808",2,0,No Missing Data
3,Industries Mapping,422,2,0,industry_name (34)
4,Companies,"24,473",10,0,"name (1), description (297), company_size (277..."


In [23]:
for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f" DATASET: {name.upper()}")
    print(f"{'='*40}")
    
    # Display the standard info
    df.info()
    
    print("-" * 40)
    display(df.head(2))


 DATASET: JOB SKILLS
<class 'pandas.DataFrame'>
RangeIndex: 213768 entries, 0 to 213767
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   job_id     213768 non-null  int64
 1   skill_abr  213768 non-null  str  
dtypes: int64(1), str(1)
memory usage: 3.3 MB
----------------------------------------


,job_id,skill_abr
0,3884428798,MRKT
1,3884428798,PR



 DATASET: SKILLS MAPPING
<class 'pandas.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   skill_abr   35 non-null     str  
 1   skill_name  35 non-null     str  
dtypes: str(2)
memory usage: 692.0 bytes
----------------------------------------


,skill_abr,skill_name
0,ART,Art/Creative
1,DSGN,Design



 DATASET: JOB INDUSTRIES
<class 'pandas.DataFrame'>
RangeIndex: 164808 entries, 0 to 164807
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   job_id       164808 non-null  int64
 1   industry_id  164808 non-null  int64
dtypes: int64(2)
memory usage: 2.5 MB
----------------------------------------


,job_id,industry_id
0,3884428798,82
1,3887473071,48



 DATASET: INDUSTRIES MAPPING
<class 'pandas.DataFrame'>
RangeIndex: 422 entries, 0 to 421
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   industry_id    422 non-null    int64
 1   industry_name  388 non-null    str  
dtypes: int64(1), str(1)
memory usage: 6.7 KB
----------------------------------------


,industry_id,industry_name
0,1,Defense and Space Manufacturing
1,3,Computer Hardware Manufacturing



 DATASET: COMPANIES
<class 'pandas.DataFrame'>
RangeIndex: 24473 entries, 0 to 24472
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   company_id    24473 non-null  int64  
 1   name          24472 non-null  str    
 2   description   24176 non-null  str    
 3   company_size  21699 non-null  float64
 4   state         24451 non-null  str    
 5   country       24473 non-null  str    
 6   city          24472 non-null  str    
 7   zip_code      24445 non-null  str    
 8   address       24451 non-null  str    
 9   url           24473 non-null  str    
dtypes: float64(1), int64(1), str(8)
memory usage: 1.9 MB
----------------------------------------


,company_id,name,description,company_size,state,country,city,zip_code,address,url
0,1009,IBM,"At IBM, we do more than work. We create. We cr...",7.0,NY,US,"Armonk, New York",10504,International Business Machines Corp.,https://www.linkedin.com/company/ibm
1,1016,GE HealthCare,Every day millions of people feel the impact o...,7.0,0,US,Chicago,0,-,https://www.linkedin.com/company/gehealthcare


In [25]:
merged_skills = job_skills.merge(skills, on='skill_abr', how='left')
skills_aggregated = merged_skills.groupby('job_id')['skill_name'].apply(
    lambda x: ', '.join(x.dropna().unique())
).reset_index()
skills_aggregated.rename(columns={'skill_name': 'explicit_required_skills'}, inplace=True)

industries['industry_name'] = industries['industry_name'].fillna('Unknown Industry')
merged_industries = job_industries.merge(industries, on='industry_id', how='left')
industries_aggregated = merged_industries.groupby('job_id')['industry_name'].apply(
    lambda x: ', '.join(x.unique())
).reset_index()

companies['name'] = companies['name'].fillna('Confidential Company')
companies['description'] = companies['description'].fillna('')
companies.rename(columns={'description': 'company_description'}, inplace=True)


companies['company_size'] = companies['company_size'].fillna('Not Specified').astype(str)

# Clean up web scraping location artifacts
companies['state'] = companies['state'].replace(['0', '-'], 'Not Specified').fillna('Not Specified')
companies['city'] = companies['city'].replace(['0', '-'], 'Not Specified').fillna('Not Specified')


In [26]:
display(skills_aggregated.head(3))
print("-"*20)
display(industries_aggregated.head(3))
print("-"*20)
display(companies.head(3))

,job_id,explicit_required_skills
0,921716,"Marketing, Sales"
1,1218575,Health Care Provider
2,1829192,Health Care Provider


--------------------


,job_id,industry_name
0,921716,Real Estate
1,1218575,Hospitals and Health Care
2,2264355,Religious Institutions


--------------------


,company_id,name,company_description,company_size,state,country,city,zip_code,address,url
0,1009,IBM,"At IBM, we do more than work. We create. We cr...",7.0,NY,US,"Armonk, New York",10504,International Business Machines Corp.,https://www.linkedin.com/company/ibm
1,1016,GE HealthCare,Every day millions of people feel the impact o...,7.0,Not Specified,US,Chicago,0,-,https://www.linkedin.com/company/gehealthcare
2,1025,Hewlett Packard Enterprise,Official LinkedIn of Hewlett Packard Enterpris...,7.0,Texas,US,Houston,77389,1701 E Mossy Oaks Rd Spring,https://www.linkedin.com/company/hewlett-packa...


In [ ]:
salaries = pd.read_csv('Data/jobs/salaries.csv')
benefits = pd.read_csv('Data/jobs/benefits.csv')
comp_specialities = pd.read_csv('Data/companies/company_specialities.csv')
comp_industries = pd.read_csv('Data/companies/company_industries.csv')
employee_counts = pd.read_csv('Data/companies/employee_counts.csv')

display(salaries)
print("-"*20)
display(benefits)
print("-"*20)
display(comp_specialities)
print("-"*20)
display(comp_industries)
print("-"*20)
display(employee_counts)

In [ ]:
raw_metadata_datasets = {
    "Raw Salaries": salaries,
    "Raw Benefits": benefits,
    "Raw Company Specialities": comp_specialities,
    "Raw Company Industries": comp_industries,
    "Raw Employee Counts": employee_counts
}

metadata_dq_report = generate_quality_report(raw_metadata_datasets)
display(metadata_dq_report)

print("\n1. Salaries:")
display(salaries.head(3))

print("\n2. Benefits:")
display(benefits.head(3))

print("\n3. Company Specialities:")
display(comp_specialities.head(3))

print("\n4. Company Industries:")
display(comp_industries.head(3))

print("\n5. Employee Counts:")
display(employee_counts.head(3))